In [2]:
# NAAnoBot training prototype

In [12]:
import torch
import numpy as np
import pandas as pd
%matplotlib inline
from pathlib import Path

In [4]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

RADIAL_SEQUENCES = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
RADIAL_SEQUENCES.set_index("PDB_ID", inplace=True)

MOLECULAR_FINGERPRINTS = pd.read_pickle(DATAPATH / "molfp_df.pkl")
MOLECULAR_FINGERPRINTS.set_index("smiles_str", inplace=True)

TRAIN_POINTERS = pd.read_parquet(DATAPATH / "training_pointers.parquet")
TEST_POINTERS = pd.read_parquet(DATAPATH / "test_pointers.parquet")

In [5]:
from src.nano_maker.modules.nAAno.naanoeng import NAAnoEng, encoder_check

In [8]:
# protein feature engineering:
# spatial data + physicochemical data
# when you pass an AA seq through ESM2, you get back a matrix of shape:
# [sequence length x embedding dimension] - big problem
# each row is a per-residue embedding, a vector that encodes that AA's...
# - identity, local chemical environment, inferred structural context as a vector
# add parts of GEM to append a physiochemical vector on top of spatial one
# - charge, hydrophobicity, pKa, H-bond capacity, .etc
# GEM properties = nAAno_token

# I don't think we need a module for tokenizing, this isn't NLP
# put stuff in nAAno_library for physicochemical analysis

# TODO: ramachandran freedom feature

from src.nano_maker.modules.nAAno.naanolibrary import *

class NAAnoEng:
    """Run this everytime we need a new set of feature vectors"""
    def __init__(self, verbose=False):
        self.verbose = verbose

    def initialize(self):
        self.nAAno_vectors = {aa_id: build_nAAnoVector(aa_id) for aa_id in AA_IDS}

        if self.verbose:
            print("NAAnoEng initialized")
        return True

    def get_aa_id(self, naano_vector):
        aa_id = None
        for code, key in self.nAAno_vectors.items():
            if key == naano_vector:
                aa_id = code
                break

        if aa_id is None:
            raise ValueError(f"Feature vector {naano_vector} presented does not exist")

        return aa_id

    def get_nAAnovector(self, aa_id):
        return self.nAAno_vectors[aa_id]


def build_nAAnoVector(aa_id: str):
    """
    use this in these two scenarios:
    \n    - generating embeddings after updating feature vector
    \n    - inference
    :param aa_id: single letter amino acid reference code
    :returns: feature vector representing a given (valid) amino acid
    """
    # sanity check
    if aa_id not in AA_IDS:
        raise ValueError(f"{aa_id} not in valid AA ID list")

    # embedding scheme, MAKE SURE TO UPDATE THIS IF YOU EVER UPDATE NAANOLIBRARY
    naano_vector = [
        MOLECULAR_WEIGHTS[aa_id],
        NET_CHARGES[aa_id],
        ISOELECTRIC_PTS[aa_id],
        HYDROPHOBICITY_IDXS[aa_id],
        HALF_LIFE[aa_id],
    ]
    naano_vector += FUNCTIONAL_FP[aa_id]
    return naano_vector


def encoder_check(verbose=True):
    module = NAAnoEng(verbose=True)
    module.initialize()
    for aa_code, aa_vect in module.nAAno_vectors.items():
        print(f"{aa_code} -- {aa_vect}")

    # check decoder and encoder
    for aa in AA_IDS:
        aa_str = aa
        aa_emb = module.get_nAAnovector(aa)
        if aa_str == module.get_aa_id(aa_emb):
            if verbose:
                print(f"{aa_str}: str <-> vect aligned")
        else:
            raise ValueError(f"Ensure {aa} in nAAno_library is up to date")
encoder_check()  # note: all good

NAAnoEng initialized
A -- [0.09, 0, 6.02, 41, 4.4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
C -- [0.12, 0, 5.02, 49, 1.2, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]
D -- [0.13, -1, 2.98, -55, 1.1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
E -- [0.15, -1, 3.22, -31, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
F -- [0.17, 0, 5.48, 100, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
G -- [0.08, 0, 5.97, 0, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
H -- [0.16, 0, 7.59, 8, 3.5, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
I -- [0.13, 0, 5.98, 100, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
K -- [0.15, 1, 9.47, -23, 1.3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
L -- [0.13, 0, 5.98, 97, 5.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
M -- [0.15, 0, 5.75, 74, 30, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
N -- [0.13, 0, 5.41, -28, 1.4, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
P -- [0.12, 0, 6.3, -46, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Q -- [0.15, 0, 5.65, -10, 0.8, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
R -- [0.17, 1,

In [13]:
from torch.utils.data import Dataset
import pyarrow.parquet as pq

# note: all coordinates are now on the spherical system
# doesn't need to predict an "end" since that's covered by Skeleton
# all it has to do is predict the next valid suitable biochemical environment

class NAAnoDataset(Dataset):
    def __init__(self, pointer_path, smiles_molfp, pdb_radial,
                 block_size, max_angstroms, n_features):

        self.pointer = pq.ParquetFile(pointer_path)
        self.table = pq.read_table(pointer_path, memory_map=True)

        self.smiles_col = self.table.column("SMILES").combine_chunks()
        self.pdb_col = self.table.column("PDB_HIT").combine_chunks()
        self.window_col = self.table.column("WINDOW_IDX").combine_chunks()

        self.smiles_molfp = smiles_molfp
        self.pdb_radial = pdb_radial

        self.block_size = block_size
        self.max_angstroms = max_angstroms
        self.n_features = n_features

    def __len__(self):
        return len(self.table)

    def __getitem__(self, idx):
        smiles = self.smiles_col[idx].as_py()
        pdb_id = self.pdb_col[idx].as_py()
        target_idx = self.window_col[idx].as_py()

        # context
        molecular_fingerprint = self.smiles_molfp.loc[smiles, "map4_fp"]
        radial_sequence = self.pdb_radial.loc[pdb_id, "radial_sequence"]
        vector_X, vector_Y = self.naano_XY(radial_sequence, target_idx)

        return (molecular_fingerprint,
                torch.tensor(vector_X, dtype=torch.float32),
                torch.tensor(vector_Y, dtype=torch.float32))


    def naano_XY(self, radial_sequence, target_idx):
        """
        ok think slightly harder about this
        - we want the relative vector to the target coordinate
        - get context first, record target Y when found and then break
        - calculate relative coordinates and append to biochemical vectors after
        :param radial_sequence:
        :param target_idx:
        :return:
        """
        r_pad = int(self.max_angstroms)
        az_pad = 0
        pl_pad = 0

        coord_context = [[r_pad, az_pad, pl_pad] for _ in range(self.block_size)]
        nAAno_context = [[-1 for _ in range(self.n_features)] for _ in range(self.block_size)]

        coord_Y = None  # what we use to calculate the relative vectors
        naano_Y = None  # this is what our model needs to predict

        append_state = True
        for idx in range(len(radial_sequence) - 1):  # subtract the VOID token out, we already have a pad for it
            nAAno_token = radial_sequence[idx][0]  # biochemical vector
            coords = radial_sequence[idx][1]  # convert back to [x, y, z]

            if idx == target_idx:
                coord_Y = coords
                naano_Y = nAAno_token
                append_state = False

            if append_state == False:
                break
            else:
                coord_context = coord_context[1:]+[coords]
                nAAno_context = nAAno_context[1:]+[nAAno_context]

        # create final context matrix
        naano_X = []

        # go through coordinate context (iterate through range)
        # calculate and add relative coordinates concat. with nAAnovectors
        for idx in range(len(coord_context)):
            coord_X = coord_context[idx]
            # construct augmented relative coordinate vector then concat with nAAno_token
            # XYZ
            relative_vect = self.sph_to_xyz(coord_X) - self.sph_to_xyz(coord_Y)  # 3
            euclidean_dist = torch.norm(relative_vect) + 1e-8   # 1
            unit_dir = relative_vect / euclidean_dist   # 1

            # spherical
            Xr, Xaz, Xpl = coord_X
            Yr, Yaz, Ypl = coord_Y
            r_diff = Yr - Xr    # 1
            az_diff = self.angle_diff(Xaz, Yaz)   # 1
            pl_diff = self.angle_diff(Xpl, Ypl)   # 1

            naano_X.append(torch.stack([relative_vect, euclidean_dist, unit_dir,
                                        r_diff, az_diff, pl_diff]))   # + 7

        return naano_X, naano_Y


    @staticmethod
    def sph_to_xyz(spherical_vector):
        r, az, pl = spherical_vector
        x = r * np.sin(pl) * np.cos(az)
        y = r * np.sin(pl) * np.sin(az)
        z = r * np.cos(pl)
        return [x.item(), y.item(), z.item()]

    @staticmethod
    def angle_diff(aX, aY):
        return ((torch.cos(aX) - torch.cos(aY))**2 + (torch.sin(aX) - torch.sin(aY))**2).mean()


In [ ]:
# Generates a suitable biochemical environment for a SMILES given nAAnoVector context
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

# NAANOBOT MODEL
class NAAnoBot(nn.Module):
    def __init__(self, n_embd, n_head, n_layers, block_size,
                 map4_res, n_features,
                 dropout):
        super().__init__()
        self.block_size = block_size
        self.map4_res = map4_res
        self.n_features = n_features

        self.nAAno_project = nn.Linear(n_features, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.mol_encoder = MolecularEncoder(n_embd, map4_res, dropout)

        self.stacks = nn.ModuleList([Stack(n_embd, n_head, block_size, dropout) for _ in range(n_layers)])

        # OUTPUT HEAD -> outputs coordinates
        self.ln_f = nn.LayerNorm(n_embd)
        self.v_head = nn.Linear(n_embd, 21)


    def forward(self, bio_context, map4_enc, targets=None):
        """
        bio_context will be [n_batch, block_size, n_features + 3]
        targets is [n_batch, 1, n_features]
        map4_enc -> unsure how I'm going to encode this
        """
        B, T, C = bio_context.shape
        coordinate_emb = self.c_project(bio_context.float())
        pos_emb = self.pos_emb(torch.arange(T, device=bio_context.device))
        x = coordinate_emb + pos_emb

        mol_emb = self.mol_encoder(map4_enc.float()).unsqueeze(1)

        for stack in self.stacks:
            x = stack(x, mol_emb)

        r, az, pl = self.c_head(self.ln_f(x[:, -1, :])).unbind(dim=1)
        r = F.softplus(r)                 # clip off negative values
        az = torch.tanh(az) * torch.pi    # prevent outputs from being nonsensical angles
        pl = torch.sigmoid(pl) * torch.pi

        output = torch.stack([r, az, pl], dim=1)

        # circular loss instead of regular coordinate MSE
        # doesn't make sense to convert to xyz so I'll use circular loss and radius MSE
        loss = None
        if targets is not None:
            # figure this out after
            loss = 42

        return output, loss


    def generate(self, map4_enc, max_AAs=130):
        # largest protein pocket in dataset was 107
        map4_enc = map4_enc.to(next(self.parameters()).device)

        # may 26 -> empty outer context pad / "token" = max_angstroms, neutral azimuth, neutral polar
        # -> inner stop "token" is 0 0 0
        # ok changing this after downloading the weights doesn't matter - separate from inference logic
        # max angstroms is actually ~5 angstroms past the max in the dataset, so the relationship is still clear
        coord_context = torch.tensor([[self.max_angstroms, 0, 0] for _ in range(self.block_size)]).unsqueeze(0).to(map4_enc.device)

        coord_out = []

        for _ in range(max_AAs):
            next_coord, _ = self.forward(coord_context, map4_enc)
            coord_out.append(next_coord.detach())
            coord_context = torch.cat([coord_context[:, 1:, :], next_coord.unsqueeze(1)], dim=1)

        return torch.stack(coord_out, dim=1).squeeze(0)

# most of the stuff below is from the nanoGPT file except the nanogpt and molecular encoder
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

        # added in a causal mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute affinities
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5
        attn_wts = attn_wts.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighed aggregation
        values = self.value(x)
        output = attn_wts @ values
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coordinate_emb, molecular_emb):
        q = self.query(coordinate_emb)
        k = self.key(molecular_emb)
        v = self.value(molecular_emb)

        head_size = q.shape[-1]

        c_attn = q @ k.transpose(-2, -1) * head_size**-0.5
        c_attn = F.softmax(c_attn, dim=-1)
        c_attn = self.dropout(c_attn)

        return c_attn @ v

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,
                                                       head_size,
                                                       dropout) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coord_emb, mol_emb):
        out = torch.cat([h(coord_emb, mol_emb) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out


class FeedForwardNet(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Stack(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.satt_head = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForwardNet(n_embd, dropout)
        self.catt_head = MultiHeadCrossAttention(n_embd, n_head, dropout)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, mol_emb):
        x = x + self.satt_head(self.ln1(x))
        x = x + self.catt_head(self.ln2(x), mol_emb)
        x = x + self.ffwd(self.ln3(x))
        return x


# previously was way too simple of an MLP, added in more layers
# consider upgrading to use ChemBERTa and save embeddings as vectors to feed in to cross-attention
class MolecularEncoder(nn.Module):
    def __init__(self, n_embd, map4_res, dropout):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Linear(map4_res, map4_res),
            nn.LayerNorm(map4_res),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.check1 = nn.Linear(map4_res, map4_res // 2)

        self.block2 = nn.Sequential(
            nn.Linear(map4_res // 2, map4_res // 2),
            nn.LayerNorm(map4_res // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.check2 = nn.Linear(map4_res // 2, n_embd)
        self.residual1 = nn.Linear(map4_res, map4_res // 2)

        self.block3 = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.LayerNorm(n_embd),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.residual2 = nn.Linear(map4_res // 2, n_embd)


        self.output = nn.LayerNorm(n_embd)

    def forward(self, x):
        x1 = self.block1(x) + x
        x1_down = self.check1(x1)

        x2 = self.block2(x1_down) + self.residual1(x1)
        x2_down = self.check2(x2)

        x3 = self.block3(x2_down) + self.residual2(x2)

        return self.output(x3)